# 01 · Perfilado de fuentes — Cloud Provider Analytics
**Minería de Datos II · ISTEA · 2C 2026 · Proyecto integrador — Primera entrega**

Objetivo: perfilar las 8 fuentes de Landing para fundamentar el inventario de fuentes, el análisis 5V,
las reglas de calidad y las decisiones de arquitectura (patrón, watermark, anomalías).

Principios del perfilado:
- **Landing es inmutable**: solo se lee, nunca se modifica.
- Los CSV se leen **todo como string** (sin `inferSchema`) para medir la casteabilidad real de cada columna,
  igual que hará la ingesta a Bronze con esquema explícito.
- El código es compatible con **Spark 3.5 y 4.x** (casts vía `try_cast` en SQL, sin depender del modo ANSI).

| Bloque | Contenido |
|---|---|
| 0 | Setup y configuración |
| 1 | Estructura de los CSV: nulos, cardinalidad, casteabilidad |
| 2 | Semántica de maestros: rangos, billing multimoneda, coherencia temporal, integridad referencial |
| 3 | Eventos de uso (JSONL): evolución de esquema, calidad, costos |
| 4 | Simulación de streaming: impacto del watermark |
| 5 | Inventario de fuentes y síntesis de implicancias de diseño |

## 0 · Setup

In [1]:
# Colab: PySpark suele venir preinstalado; si no, se instala una versión compatible con Java 11/17
try:
    import pyspark
except ImportError:
    %pip install -q "pyspark==3.5.5"
    import pyspark
print("PySpark", pyspark.__version__)

PySpark 4.2.0


In [2]:
import os, zipfile
# ---- CONFIGURACIÓN ----------------------------------------------------------
# En Colab: subir los 7 CSV + usage_events_stream.zip a DATA_DIR (o montar Drive y apuntar ahí).
# from google.colab import drive; drive.mount('/content/drive')
# DATA_DIR = "/content/drive/MyDrive/cpa/landing"
DATA_DIR = os.environ.get("CPA_DATA_DIR", "/content/cpa/landing")
EVENTS_DIR = os.path.join(DATA_DIR, "usage_events_stream")
# ------------------------------------------------------------------------------
zip_path = os.path.join(DATA_DIR, "usage_events_stream.zip")
if not os.path.isdir(EVENTS_DIR) and os.path.exists(zip_path):
    zipfile.ZipFile(zip_path).extractall(DATA_DIR)   # se extrae una copia; el zip original no se toca
print("CSV:", sorted(f for f in os.listdir(DATA_DIR) if f.endswith(".csv")))
print("JSONL:", len([f for f in os.listdir(EVENTS_DIR) if f.endswith(".jsonl")]), "archivos")

CSV: ['billing_monthly.csv', 'customers_orgs.csv', 'marketing_touches.csv', 'nps_surveys.csv', 'resources.csv', 'support_tickets.csv', 'users.csv']
JSONL: 120 archivos


In [3]:
from pyspark.sql import SparkSession, functions as F, Window as W
spark = (SparkSession.builder.master("local[*]").appName("CPA-01-perfilado")
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.sql.session.timeZone", "UTC")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

NULL_TOKENS = ["", "null", "none", "nan", "na", "n/a", "-"]
CSV_SOURCES = ["customers_orgs", "users", "resources", "support_tickets",
               "marketing_touches", "nps_surveys", "billing_monthly"]

def read_csv_raw(name):
    """Lectura cruda: todo string, sin inferencia. Mismo criterio que Bronze."""
    return spark.read.option("header", True).option("inferSchema", False).csv(f"{DATA_DIR}/{name}.csv")

def num(c):   # cast seguro, compatible 3.5 / 4.x
    return F.expr(f"try_cast(`{c}` as double)")

raw = {n: read_csv_raw(n) for n in CSV_SOURCES}
for n, df in raw.items():
    df.cache()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/02 22:21:53 WARN Utils: Your hostname, vm, resolves to a loopback address: 127.0.0.1; using 192.0.2.2 instead (on interface eth0)
26/10/02 22:21:53 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


/usr/local/lib/python3.12/dist-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


26/10/02 22:21:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 1 · Estructura de los CSV

In [4]:
import pandas as pd
pd.set_option("display.max_colwidth", 60); pd.set_option("display.width", 200)

def profile(name):
    df = raw[name]; n = df.count(); out = []
    for c in df.columns:
        col = F.col(f"`{c}`")
        is_null = col.isNull() | F.lower(F.trim(col)).isin(NULL_TOKENS)
        r = df.agg(F.sum(is_null.cast("int")).alias("nulls"),
                   F.countDistinct(col).alias("distinct"),
                   F.sum((~is_null & num(c).isNotNull()).cast("int")).alias("num_ok"),
                   F.sum((~is_null & F.expr(f"try_to_timestamp(`{c}`)").isNotNull()).cast("int")).alias("ts_ok")).first()
        nn = n - r["nulls"]
        ej = [x[0] for x in df.where(~is_null).select(col).distinct().limit(3).collect()]
        out.append({"columna": c, "nulos": r["nulls"], "nulos_%": round(100*r["nulls"]/n, 1),
                    "distintos": r["distinct"], "castea_num_%": round(100*r["num_ok"]/nn) if nn else None,
                    "castea_fecha_%": round(100*r["ts_ok"]/nn) if nn else None, "ejemplos": ej})
    print(f"=== {name}: {n} filas · {len(df.columns)} columnas ===")
    display(pd.DataFrame(out))

for n in CSV_SOURCES:
    profile(n)

=== customers_orgs: 80 filas · 11 columnas ===


,columna,nulos,nulos_%,distintos,castea_num_%,castea_fecha_%,ejemplos
0,org_id,0,0.0,80,0,0,"[org_1t2tala7, org_dzr1yxr2, org_9t84azyt]"
1,org_name,0,0.0,80,0,0,"[Delta Digital 49, Vertex Labs 35, Nimbus Cloud 76]"
2,industry,0,0.0,10,0,0,"[Education, Gaming, Energy]"
3,hq_region,0,0.0,7,0,0,"[sa-east, eu-west, us-west]"
4,plan_tier,0,0.0,4,0,0,"[free, pro, enterprise]"
5,is_enterprise,0,0.0,2,0,0,"[False, True]"
6,signup_date,0,0.0,40,0,100,"[2025-06-03, 2025-06-16, 2025-05-23]"
7,sales_rep,0,0.0,5,0,0,"[rep_a, rep_e, rep_c]"
8,lifecycle_stage,0,0.0,5,0,0,"[at_risk, churned, active]"
9,marketing_source,0,0.0,5,0,0,"[partner, ads, referral]"


=== users: 800 filas · 7 columnas ===


,columna,nulos,nulos_%,distintos,castea_num_%,castea_fecha_%,ejemplos
0,user_id,0,0.0,800,0,0,"[user_yjxrpj1e, user_qkbhb7s9, user_yt8qzpky]"
1,org_id,0,0.0,80,0,0,"[org_1t2tala7, org_dzr1yxr2, org_9t84azyt]"
2,email,0,0.0,800,0,0,"[user_1w1ht7pz@example.com, user_b2knftu2@example.com, u..."
3,role,0,0.0,6,0,0,"[ml_engineer, devops, developer]"
4,active,0,0.0,2,0,0,"[False, True]"
5,created_at,0,0.0,100,0,100,"[2025-06-03, 2025-06-16, 2025-07-15]"
6,last_login,139,17.4,110,0,100,"[2025-06-03, 2025-08-17, 2025-08-18]"


=== resources: 400 filas · 7 columnas ===


...

,columna,nulos,nulos_%,distintos,castea_num_%,castea_fecha_%,ejemplos
0,touch_id,0,0.0,1500,0,0,"[mkt_8utd9jyj, mkt_5dnsvbcq, mkt_8oveuur6]"
1,org_id,0,0.0,80,0,0,"[org_1t2tala7, org_dzr1yxr2, org_9t84azyt]"
2,campaign,0,0.0,6,0,0,"[webinar_finops, welcome, upgrade_enterprise]"
3,channel,0,0.0,4,0,0,"[ads, email, in_app]"
4,timestamp,0,0.0,120,0,100,"[2025-06-03, 2025-08-17, 2025-08-18]"
5,clicked,0,0.0,2,0,0,"[False, True]"
6,converted,0,0.0,2,0,0,"[False, True]"


=== nps_surveys: 92 filas · 4 columnas ===


,columna,nulos,nulos_%,distintos,castea_num_%,castea_fecha_%,ejemplos
0,org_id,0,0.0,60,0,0,"[org_dzr1yxr2, org_9t84azyt, org_7e8g8jdp]"
1,survey_date,0,0.0,57,0,100,"[2025-06-03, 2025-06-16, 2025-07-21]"
2,nps_score,19,20.7,41,100,0,"[50.0, 22.0, 38.0]"
3,comment,10,10.9,6,0,0,"[Complex billing, Great support, Too expensive]"


=== billing_monthly: 240 filas · 8 columnas ===


,columna,nulos,nulos_%,distintos,castea_num_%,castea_fecha_%,ejemplos
0,invoice_id,0,0.0,240,0,0,"[inv_2ujz4n1x, inv_mymmhlk1, inv_2798p7pr]"
1,org_id,0,0.0,80,0,0,"[org_1t2tala7, org_dzr1yxr2, org_9t84azyt]"
2,month,0,0.0,3,0,100,"[2025-06-01, 2025-08-01, 2025-07-01]"
3,subtotal,0,0.0,240,100,0,"[1851.84, 306.75, 1371.34]"
4,credits,137,57.1,100,100,0,"[13.87, 59.87, 5.42]"
5,taxes,0,0.0,240,100,0,"[121.82, 182.93, 158.75]"
6,currency,0,0.0,3,0,0,"[ARS, EUR, USD]"
7,exchange_rate_to_usd,0,0.0,208,100,0,"[1.13637, 1.01834, 1.00331]"


### Lectura del bloque 1
- **No hay tipos ambiguos en los maestros**: toda columna numérica castea al 100%. Aun así Bronze usa esquema explícito con cast seguro (`try_cast`) porque la consigna lo exige y protege ante cargas futuras.
- **Nulos con significado de negocio (no son error):** `support_tickets.resolved_at` (24% → ticket abierto), `billing_monthly.credits` (57% → sin crédito, se imputa 0), `users.last_login` (17% → nunca ingresó).
- **Nulos que son faltantes reales:** `support_tickets.csat` (25%), `customers_orgs.nps_score` (14%), `nps_surveys.nps_score` (21%). No se imputan: se excluyen de promedios y se reporta la cobertura.
- **`resources.tags_json` (21% nulo) viene mal formado** → ver bloque 2.

## 2 · Semántica de los maestros

### 2.1 Rangos de scores

In [5]:
for name, c in [("customers_orgs", "nps_score"), ("nps_surveys", "nps_score"), ("support_tickets", "csat")]:
    r = raw[name].select(num(c).alias("v")).agg(F.min("v"), F.max("v"), F.round(F.avg("v"), 1)).first()
    print(f"{name}.{c}: min={r[0]} max={r[1]} avg={r[2]}")
raw["support_tickets"].groupBy(num("csat").alias("csat")).count().orderBy("csat").show()

customers_orgs.nps_score: min=-38.0 max=101.0 avg=18.7
nps_surveys.nps_score: min=-16.0 max=68.0 avg=26.2


support_tickets.csat: min=0.0 max=7.0 avg=3.3


+----+-----+
|csat|count|
+----+-----+
|NULL|  254|
| 0.0|   11|
| 1.0|   45|
| 2.0|  127|
| 3.0|  242|
| 4.0|  197|
| 5.0|   95|
| 6.0|   28|
| 7.0|    1|
+----+-----+



`nps_score` de org llega a **101** (fuera del rango NPS −100..100) → regla de validez. `csat` va de 0 a 7: la escala no está documentada; se asume 0–7 y se registra como supuesto a validar.

### 2.2 Billing multimoneda

In [6]:
b = raw["billing_monthly"].select("invoice_id", "org_id", "month", "currency",
        *[num(c).alias(c) for c in ["subtotal", "credits", "taxes", "exchange_rate_to_usd"]])
b.groupBy("currency").agg(F.count("*").alias("n"),
    F.min("exchange_rate_to_usd").alias("fx_min"), F.max("exchange_rate_to_usd").alias("fx_max"),
    F.round(F.avg("subtotal"), 0).alias("subtotal_avg"), F.min("subtotal").alias("subtotal_min"),
    F.sum((F.col("subtotal") < 0).cast("int")).alias("subtotal_neg")).orderBy("currency").show()
print("Facturas por org:", b.groupBy("org_id").count().groupBy("count").count().collect())
print("Ratio taxes/subtotal (min, max):", b.select((F.col("taxes")/F.col("subtotal")).alias("r")).agg(F.round(F.min("r"),3), F.round(F.max("r"),3)).first())
print("Créditos negativos o mayores al subtotal:", b.where((F.col("credits") < 0) | (F.col("credits") > F.col("subtotal"))).count())
print("Orgs con más de una moneda en 3 meses:", b.groupBy("org_id").agg(F.countDistinct("currency").alias("k")).where("k > 1").count(), "de 80")

+--------+---+-------+-------+------------+------------+------------+
|currency|  n| fx_min| fx_max|subtotal_avg|subtotal_min|subtotal_neg|
+--------+---+-------+-------+------------+------------+------------+
|     ARS| 51|0.00133|0.00162|       807.0|    -1355.74|           2|
|     EUR| 29| 0.9981|1.19808|       814.0|     -1095.1|           1|
|     USD|160|0.85463|1.11791|       671.0|    -1671.83|          10|
+--------+---+-------+-------+------------+------------+------------+



Facturas por org: [Row(count=3, count=80)]


Ratio taxes/subtotal (min, max): Row(round(min(r), 3)=-0.21, round(max(r), 3)=0.21)
Créditos negativos o mayores al subtotal: 9


Orgs con más de una moneda en 3 meses: 50 de 80


**Hallazgos críticos para FinOps (mart `revenue_by_org_month`, consulta 4):**
- Subtotales **negativos** en las tres monedas; los impuestos acompañan el signo (ratio ±0,21) → posibles notas de crédito. Decisión pendiente: ¿son ajustes válidos o errores?
- Filas en **USD con `exchange_rate_to_usd` entre 0,85 y 1,12** (debería ser 1,0) → el FX o la moneda no son confiables. Propuesta: para USD forzar FX = 1 y flaguear la discrepancia.
- **50 de 80 orgs facturan en más de una moneda** en 3 meses: inusual; se documenta como supuesto (moneda a nivel factura, no de cliente).
- Créditos negativos o mayores al subtotal (9 casos) → regla de calidad.

### 2.3 `tags_json` en resources

In [7]:
t = raw["resources"].where(F.col("tags_json").isNotNull()).select("tags_json")
parseables = t.where(F.from_json(F.regexp_replace("tags_json", '""', '"'), "array<string>").isNotNull()).count()
print(f"Parseables como JSON array: {parseables} de {t.count()}")
t.groupBy("tags_json").count().orderBy(F.desc("count")).show(20, truncate=False)
# Alternativa robusta para Silver: extraer el par key:value con regex
t.select(F.regexp_extract("tags_json", r"([a-z]+):([a-z]+)", 0).alias("tag")).groupBy("tag").count().orderBy("tag").show()

Parseables como JSON array: 0 de 317


+------------------------+-----+
|tags_json               |count|
+------------------------+-----+
|"[""env:dev""           |37   |
|"[""pii:true""          |32   |
|"[""team:core""         |30   |
|"[""costcenter:alpha""  |25   |
|"[""team:ml""           |24   |
|"[""env:prod""          |23   |
|"[""backup:daily""      |21   |
|"[""costcenter:beta""   |19   |
|"[""team:ml""]"         |17   |
|"[""costcenter:beta""]" |15   |
|"[""env:prod""]"        |15   |
|"[""team:core""]"       |14   |
|"[""pii:true""]"        |13   |
|"[""backup:daily""]"    |12   |
|"[""env:dev""]"         |11   |
|"[""costcenter:alpha""]"|9    |
+------------------------+-----+



+----------------+-----+
|             tag|count|
+----------------+-----+
|    backup:daily|   33|
|costcenter:alpha|   34|
| costcenter:beta|   34|
|         env:dev|   48|
|        env:prod|   38|
|        pii:true|   45|
|       team:core|   44|
|         team:ml|   41|
+----------------+-----+



**0% parseable como JSON**: doble escape de CSV y ~la mitad truncado sin `]`. Decisión para Silver: **extraer `key:value` con regex** (cubre el 100%) en lugar de `from_json`. El tag `pii:true` es insumo directo para la clasificación de sensibilidad en gobierno.

### 2.4 Coherencia temporal y ventana de datos

In [8]:
for name, c in [("customers_orgs", "signup_date"), ("users", "created_at"), ("resources", "created_at"),
                ("support_tickets", "created_at"), ("marketing_touches", "timestamp"), ("nps_surveys", "survey_date"),
                ("billing_monthly", "month")]:
    r = raw[name].agg(F.min(c), F.max(c)).first(); print(f"{name}.{c}: {r[0]} -> {r[1]}")

tk, us, og = raw["support_tickets"], raw["users"], raw["customers_orgs"]
print("\ntickets resolved_at < created_at:", tk.where(F.to_date("resolved_at") < F.to_date("created_at")).count())
print("users last_login < created_at:", us.where(F.to_date("last_login") < F.to_date("created_at")).count())
print("users creados antes del signup de su org:",
      us.join(og.select("org_id", "signup_date"), "org_id").where(F.to_date("created_at") < F.to_date("signup_date")).count())
print("\nTickets abiertos por severidad:"); tk.where(F.col("resolved_at").isNull()).groupBy("severity").count().show()
print("SLA breach rate por severidad:")
tk.groupBy("severity").agg(F.count("*").alias("n"),
    F.round(F.avg(F.expr("try_cast(sla_breached as boolean)").cast("int")), 3).alias("breach_rate")).orderBy("breach_rate").show()

customers_orgs.signup_date: 2025-05-04 -> 2025-07-02


users.created_at: 2025-05-04 -> 2025-08-11


resources.created_at: 2025-05-04 -> 2025-08-21


support_tickets.created_at: 2025-05-09 -> 2025-08-31


marketing_touches.timestamp: 2025-05-04 -> 2025-08-31


...


tickets resolved_at < created_at: 0


users last_login < created_at: 232


users creados antes del signup de su org: 249

Tickets abiertos por severidad:


+--------+-----+
|severity|count|
+--------+-----+
|critical|   18|
|     low|   92|
|    high|   50|
|  medium|   80|
+--------+-----+

SLA breach rate por severidad:


+--------+---+-----------+
|severity|  n|breach_rate|
+--------+---+-----------+
|critical| 56|      0.036|
|     low|412|      0.087|
|    high|204|      0.098|
|  medium|328|      0.113|
+--------+---+-----------+



- Ventana común de maestros: **mayo a agosto 2025**; billing cubre jun–ago; eventos (bloque 3) jul–ago.
- `users`: 232 con `last_login < created_at` y 249 creados antes del signup de su org → **flag de calidad, no quarantine** (rol y actividad siguen siendo útiles).
- **SLA invertido**: `critical` tiene la menor tasa de breach (3,6%) y `medium` la mayor (11,3%). A validar: comportamiento real (priorización) o artefacto del generador. Insight para el dominio Soporte.

### 2.5 Integridad referencial y categóricas

In [9]:
orgs_ids = og.select("org_id").distinct()
for name in ["users", "resources", "support_tickets", "marketing_touches", "nps_surveys", "billing_monthly"]:
    print(f"{name}: org_id huérfanos = {raw[name].join(orgs_ids, 'org_id', 'left_anti').count()}")
print()
for name, c in [("customers_orgs","plan_tier"), ("customers_orgs","hq_region"), ("customers_orgs","lifecycle_stage"),
                ("resources","service"), ("resources","region"), ("support_tickets","category"), ("support_tickets","severity")]:
    print(f"{name}.{c}:", sorted(r[0] for r in raw[name].select(c).distinct().collect()))
print("\nplan_tier vs is_enterprise:"); og.groupBy("plan_tier", "is_enterprise").count().orderBy("plan_tier", "is_enterprise").show()

users: org_id huérfanos = 0


resources: org_id huérfanos = 0


support_tickets: org_id huérfanos = 0


marketing_touches: org_id huérfanos = 0


nps_surveys: org_id huérfanos = 0


...

customers_orgs.plan_tier: ['enterprise', 'free', 'pro', 'standard']
customers_orgs.hq_region: ['ap-northeast', 'ap-south', 'eu-central', 'eu-west', 'sa-east', 'us-east', 'us-west']


customers_orgs.lifecycle_stage: ['active', 'at_risk', 'churned', 'lead', 'prospect']
resources.service: ['analytics', 'compute', 'database', 'genai', 'networking', 'storage']


resources.region: ['ap-northeast', 'ap-south', 'eu-central', 'eu-west', 'sa-east', 'us-east', 'us-west']
support_tickets.category: ['availability', 'billing', 'integration', 'performance', 'security', 'usability']


support_tickets.severity: ['critical', 'high', 'low', 'medium']

plan_tier vs is_enterprise:


+----------+-------------+-----+
| plan_tier|is_enterprise|count|
+----------+-------------+-----+
|enterprise|        False|    9|
|enterprise|         True|    1|
|      free|        False|    6|
|      free|         True|    2|
|       pro|        False|   15|
|       pro|         True|    6|
|  standard|        False|   33|
|  standard|         True|    8|
+----------+-------------+-----+



- **Integridad referencial perfecta** (0 huérfanos) y **categóricas limpias** (sin variantes de escritura en regiones ni servicios).
- `plan_tier` vs `is_enterprise` **inconsistente** (9 orgs `enterprise` con flag False, 16 no-enterprise con flag True) → definir fuente de verdad (propuesta: `plan_tier`) y flaguear.

## 3 · Eventos de uso (`usage_events_stream/*.jsonl`)

In [10]:
ev_raw = (spark.read.option("primitivesAsString", True).json(f"{EVENTS_DIR}/*.jsonl")
          .withColumn("source_file", F.regexp_extract(F.col("_metadata.file_path"), r"(events_part_\d+)", 1)))
ev_raw.cache()
n_ev = ev_raw.count()
print(f"Eventos: {n_ev} · archivos: {ev_raw.select('source_file').distinct().count()} · columnas: {ev_raw.columns}")
print("Registros corruptos:", ev_raw.where(F.col('_corrupt_record').isNotNull()).count() if '_corrupt_record' in ev_raw.columns else 0)
print("Orgs:", ev_raw.select("org_id").distinct().count(), "· recursos:", ev_raw.select("resource_id").distinct().count())

Eventos: 43200 · archivos: 120 · columnas: ['carbon_kg', 'cost_usd_increment', 'event_id', 'genai_tokens', 'metric', 'org_id', 'region', 'resource_id', 'schema_version', 'service', 'timestamp', 'unit', 'value', 'source_file']
Registros corruptos: 0


Orgs: 80 · recursos: 400


### 3.1 Evolución de esquema v1 → v2

In [11]:
ev_raw.groupBy("schema_version").agg(F.count("*").alias("n"),
    F.sum(F.col("carbon_kg").isNotNull().cast("int")).alias("con_carbon_kg"),
    F.sum(F.col("genai_tokens").isNotNull().cast("int")).alias("con_genai_tokens"),
    F.min("timestamp").alias("ts_min"), F.max("timestamp").alias("ts_max")).orderBy("schema_version").show(truncate=False)
print("genai_tokens por servicio:"); ev_raw.where(F.col("genai_tokens").isNotNull()).groupBy("service", "schema_version").count().show()
print("Eventos genai v2 sin tokens:", ev_raw.where("service = 'genai' AND schema_version = '2' AND genai_tokens IS NULL").count())

+--------------+-----+-------------+----------------+--------------------+--------------------+
|schema_version|n    |con_carbon_kg|con_genai_tokens|ts_min              |ts_max              |
+--------------+-----+-------------+----------------+--------------------+--------------------+
|1             |10800|0            |0               |2025-07-03T00:02:00Z|2025-07-17T23:56:00Z|
|2             |32400|32400        |3132            |2025-07-18T00:01:00Z|2025-08-31T23:58:00Z|
+--------------+-----+-------------+----------------+--------------------+--------------------+

genai_tokens por servicio:


+-------+--------------+-----+
|service|schema_version|count|
+-------+--------------+-----+
|  genai|             2| 3132|
+-------+--------------+-----+



Eventos genai v2 sin tokens: 0


**Corte temporal limpio**: v1 del 03/07 al 17/07 (10.800 eventos), v2 del 18/07 al 31/08 (32.400). `carbon_kg` presente en el 100% de v2; `genai_tokens` solo en servicio genai v2 (100% de esos eventos). Compatibilización en Silver: columnas nulas en v1 + `schema_version` conservado para trazabilidad.

### 3.2 Nulos, tipos y duplicados

In [12]:
rows = []
for c in [x for x in ev_raw.columns if x not in ("source_file", "_corrupt_record")]:
    r = ev_raw.agg(F.sum(F.col(c).isNull().cast("int")).alias("nul"),
                   F.sum((F.col(c).isNotNull() & num(c).isNull()).cast("int")).alias("no_num")).first()
    rows.append({"campo": c, "nulos": r["nul"], "nulos_%": round(100*r["nul"]/n_ev, 1), "no_numericos": r["no_num"]})
display(pd.DataFrame(rows))

dup = ev_raw.groupBy("event_id").count().where("count > 1").count()
print("event_id nulos:", ev_raw.where(F.col("event_id").isNull()).count(), "· event_id duplicados:", dup)
print("value presente y unit nulo:", ev_raw.where(F.col("value").isNotNull() & F.col("unit").isNull()).count())
print("\nmetric -> unit (¿relación 1:1?):"); ev_raw.where(F.col("unit").isNotNull()).groupBy("metric").agg(F.collect_set("unit").alias("units")).show(truncate=False)
print("value nulo por metric:"); ev_raw.where(F.col("value").isNull()).groupBy("metric").count().show()

,campo,nulos,nulos_%,no_numericos
0,carbon_kg,10800,25.0,0
1,cost_usd_increment,0,0.0,0
2,event_id,0,0.0,43200
3,genai_tokens,40068,92.8,0
4,metric,0,0.0,43200
5,org_id,0,0.0,43200
6,region,0,0.0,43200
7,resource_id,0,0.0,43200
8,schema_version,0,0.0,0
9,service,0,0.0,43200


event_id nulos: 0 · event_id duplicados: 0
value presente y unit nulo: 2038

metric -> unit (¿relación 1:1?):


+----------------+----------+
|metric          |units     |
+----------------+----------+
|storage_gb_hours|[gb_hours]|
|requests        |[count]   |
|cpu_hours       |[hours]   |
+----------------+----------+

value nulo por metric:


+----------------+-----+
|          metric|count|
+----------------+-----+
|storage_gb_hours|  283|
|        requests|  399|
|       cpu_hours|  195|
+----------------+-----+



- **Sin tipos ambiguos ni duplicados** en este dataset (aunque la consigna los anuncia). Se implementan igual (cast con fallback, dedupe por `event_id`); para **demostrar idempotencia** habrá que reprocesar o inyectar duplicados de prueba.
- **`unit` nulo con `value` presente: 2.038 eventos (4,7%)** → la regla sugerida por la consigna dispara. Como `metric → unit` es **1:1**, la decisión propuesta es **imputar `unit` desde `metric` con flag** `unit_imputed`, no enviar ~5% a quarantine.
- `value` nulo en 877 eventos (2%): el costo existe pero la métrica no → se conservan para FinOps y se excluyen de métricas de uso.

### 3.3 Costos: negativos y spikes

In [13]:
ev = ev_raw.withColumn("cost", num("cost_usd_increment")).withColumn("ts", F.to_timestamp("timestamp"))
r = ev.agg(F.min("cost"), *[F.percentile_approx("cost", p) for p in (0.5, 0.99, 0.999)], F.max("cost")).first()
print("cost_usd_increment min / p50 / p99 / p99.9 / max:", [round(x, 3) for x in r])
print("Negativos:", ev.where("cost < 0").count(), "· < -0.01:", ev.where("cost < -0.01").count(), "· > 100:", ev.where("cost > 100").count())
ev.groupBy("service").agg(F.round(F.percentile_approx("cost", 0.5), 3).alias("p50"),
    F.round(F.percentile_approx("cost", 0.99), 2).alias("p99"), F.round(F.max("cost"), 1).alias("max"),
    F.sum((F.col("cost") < 0).cast("int")).alias("negativos"), F.sum((F.col("cost") > 100).cast("int")).alias("gt_100")).orderBy("service").show()

cost_usd_increment min / p50 / p99 / p99.9 / max: [-154.461, 1.004, 16.693, 103.608, 317.431]


Negativos: 216 · < -0.01: 211 · > 100: 48


+----------+-----+-----+-----+---------+------+
|   service|  p50|  p99|  max|negativos|gt_100|
+----------+-----+-----+-----+---------+------+
| analytics|1.216|11.63|249.4|       19|     8|
|   compute|1.373|13.49|302.3|       56|    26|
|  database|0.919| 8.36|149.9|       44|     9|
|     genai|2.044|19.73|317.4|       21|     5|
|networking| 0.19| 1.67| 29.6|       32|     0|
|   storage|0.354| 3.34| 63.0|       44|     0|
+----------+-----+-----+-----+---------+------+



216 costos negativos (mín. −154) y 48 spikes > 100 USD (máx. 317) sobre un p99 de 16,7. **La escala varía ~10× entre servicios** (mediana networking 0,19 vs genai 2,04) → la detección de anomalías debe ser **por servicio** (z-score robusto / MAD por servicio u org+servicio); un umbral global marcaría todo genai y nada de networking.

### 3.4 Integridad de eventos con maestros

In [14]:
res_m = raw["resources"].select("resource_id", F.col("service").alias("svc_m"), F.col("org_id").alias("org_m"))
print("org_id huérfanos:", ev.join(orgs_ids, "org_id", "left_anti").count(),
      "· resource_id huérfanos:", ev.join(res_m, "resource_id", "left_anti").count())
j = ev.join(res_m, "resource_id")
print("service evento != service recurso:", j.where(F.col("service") != F.col("svc_m")).count(),
      "· org evento != org recurso:", j.where(F.col("org_id") != F.col("org_m")).count())
print("Eventos por día (min, max, días):", ev.groupBy(F.to_date("ts")).count().agg(F.min("count"), F.max("count"), F.count("*")).first())

org_id huérfanos: 0 · resource_id huérfanos: 0


service evento != service recurso: 0 · org evento != org recurso: 0


Eventos por día (min, max, días): Row(min(count)=720, max(count)=720, count(1)=60)


## 4 · Simulación de streaming: impacto del watermark
Cada archivo `events_part_XXXX` representa un micro-lote. Si los archivos llegaran **ordenados en el tiempo**, un watermark corto funcionaría.
Se verifica el rango temporal de cada archivo y se simula la lectura con `maxFilesPerTrigger=1` en orden de nombre:
un evento es *tardío* si su `timestamp` es anterior a (máximo event-time visto en archivos previos − watermark).

In [15]:
evf = ev.withColumn("file_n", F.regexp_extract("source_file", r"(\d+)", 1).cast("int"))
pf = evf.groupBy("file_n").agg(F.min("ts").alias("ts_min"), F.max("ts").alias("ts_max"), F.count("*").alias("n"))
pf.orderBy("file_n").show(5, truncate=False)
pf.select(F.round(F.avg((F.unix_timestamp("ts_max") - F.unix_timestamp("ts_min"))/86400), 1).alias("dias_cubiertos_prom_por_archivo")).show()

pf2 = pf.withColumn("max_previo", F.max("ts_max").over(W.orderBy("file_n").rowsBetween(W.unboundedPreceding, -1)))
sim = evf.join(pf2.select("file_n", "max_previo"), "file_n").cache()
res = []
for label, secs in [("10 min", 600), ("1 hora", 3600), ("1 día", 86400), ("7 días", 7*86400), ("30 días", 30*86400), ("60 días", 60*86400)]:
    late = sim.where(F.col("max_previo").isNotNull() &
                     (F.unix_timestamp("ts") < F.unix_timestamp("max_previo") - secs)).count()
    res.append({"watermark": label, "eventos_tardios": late, "%": round(100*late/n_ev, 1)})
display(pd.DataFrame(res))

+------+-------------------+-------------------+---+
|file_n|ts_min             |ts_max             |n  |
+------+-------------------+-------------------+---+
|0     |2025-07-03 00:08:00|2025-08-31 16:32:00|360|
|1     |2025-07-03 02:38:00|2025-08-31 23:51:00|360|
|2     |2025-07-03 14:16:00|2025-08-31 20:47:00|360|
|3     |2025-07-03 00:48:00|2025-08-31 23:51:00|360|
|4     |2025-07-03 10:43:00|2025-08-31 20:40:00|360|
+------+-------------------+-------------------+---+
only showing top 5 rows


+-------------------------------+
|dias_cubiertos_prom_por_archivo|
+-------------------------------+
|                           59.7|
+-------------------------------+



,watermark,eventos_tardios,%
0,10 min,42829,99.1
1,1 hora,42801,99.1
2,1 día,42118,97.5
3,7 días,37831,87.6
4,30 días,21421,49.6
5,60 días,0,0.0


### Lectura y decisión
**Los archivos no están ordenados en el tiempo**: cada uno cubre los ~60 días completos. Con watermark de 1 hora, **el 99% de los eventos sería tardío**.

El watermark solo descarta en operadores **con estado** (agregaciones por ventana, `dropDuplicatesWithinWatermark`); un append sin estado a Bronze no pierde nada.
Un streaming "de manual" con ventanas diarias y watermark corto produciría un Gold casi vacío **sin errores visibles**.

| Opción | Descripción | Evaluación |
|---|---|---|
| A | Watermark de 60 días | Funciona, pero convierte el streaming en batch disfrazado y el estado crece sin control |
| **B** | **Streaming solo para ingesta a Bronze** (esquema explícito, dedupe idempotente con `foreachBatch` + merge por `event_id`, checkpoint); **agregaciones por event-time en batch** sobre Silver | **Elegida**: sin pérdida de datos, idempotente, y justifica Lambda con evidencia |
| C | Agregar por tiempo de procesamiento | Descartada: semánticamente incorrecta para costos diarios |

El watermark se mantiene configurado y se **mide** la proporción de late data como métrica de observabilidad (requisito de la consigna).

## 5 · Inventario de fuentes y síntesis

In [16]:
inv = [
 ("customers_orgs.csv", "1 fila por org", "org_id", 80, "batch (maestro)", "nps_score fuera de rango y nulo; plan_tier vs is_enterprise inconsistente"),
 ("users.csv", "1 fila por usuario", "user_id", 800, "batch (maestro)", "last_login < created_at; usuarios previos al signup de la org"),
 ("resources.csv", "1 fila por recurso", "resource_id", 400, "batch (maestro)", "tags_json mal formado/truncado (0% parseable)"),
 ("support_tickets.csv", "1 fila por ticket", "ticket_id", 1000, "batch (diario)", "csat nulo 25%; resolved_at nulo = abierto; SLA invertido"),
 ("marketing_touches.csv", "1 fila por interacción", "touch_id", 1500, "batch (diario)", "limpio"),
 ("nps_surveys.csv", "1 fila por encuesta (org, fecha)", "org_id + survey_date", 92, "batch (periódico)", "nps nulo 21%; solo 60 de 80 orgs"),
 ("billing_monthly.csv", "1 fila por factura (org, mes)", "invoice_id", 240, "batch (mensual)", "subtotales negativos; FX≠1 en USD; multimoneda por org"),
 ("usage_events_stream/*.jsonl", "1 fila por evento de uso", "event_id", n_ev, "streaming (micro-lotes)", "schema v1/v2; unit nulo 4,7%; costos negativos/spikes; archivos desordenados"),
]
display(pd.DataFrame(inv, columns=["fuente", "grano", "clave", "filas", "frecuencia / modo", "riesgos de calidad"]))

,fuente,grano,clave,filas,frecuencia / modo,riesgos de calidad
0,customers_orgs.csv,1 fila por org,org_id,80,batch (maestro),nps_score fuera de rango y nulo; plan_tier vs is_enterpr...
1,users.csv,1 fila por usuario,user_id,800,batch (maestro),last_login < created_at; usuarios previos al signup de l...
2,resources.csv,1 fila por recurso,resource_id,400,batch (maestro),tags_json mal formado/truncado (0% parseable)
3,support_tickets.csv,1 fila por ticket,ticket_id,1000,batch (diario),csat nulo 25%; resolved_at nulo = abierto; SLA invertido
4,marketing_touches.csv,1 fila por interacción,touch_id,1500,batch (diario),limpio
5,nps_surveys.csv,"1 fila por encuesta (org, fecha)",org_id + survey_date,92,batch (periódico),nps nulo 21%; solo 60 de 80 orgs
6,billing_monthly.csv,"1 fila por factura (org, mes)",invoice_id,240,batch (mensual),subtotales negativos; FX≠1 en USD; multimoneda por org
7,usage_events_stream/*.jsonl,1 fila por evento de uso,event_id,43200,streaming (micro-lotes),"schema v1/v2; unit nulo 4,7%; costos negativos/spikes; a..."


### Implicancias de diseño (insumo para arquitectura v1 y Data Lake)
| Hallazgo | Implicancia | Dónde se resuelve |
|---|---|---|
| Archivos de eventos desordenados en el tiempo | Streaming solo a Bronze; agregaciones event-time en batch → **Lambda** (opción B) | Arquitectura / Spark |
| Schema v1/v2 con corte limpio | Esquema unificado con columnas opcionales; conservar `schema_version` | Bronze → Silver |
| `unit` nulo con `value` (4,7%), metric→unit 1:1 | Imputar con flag, no quarantine | Regla de calidad Silver |
| Costos negativos y spikes, escala distinta por servicio | Flag `cost < -0.01` + anomalía robusta **por servicio** | Silver / `cost_anomaly_mart` |
| Billing: negativos, FX inconsistente, multimoneda | Reglas de validez + supuesto documentado sobre FX | Silver / `revenue_by_org_month` |
| `tags_json` no parseable | Extracción por regex; `pii:true` → clasificación de sensibilidad | Silver / Gobierno |
| Nulos con significado (`resolved_at`, `credits`) | Semántica explícita en diccionario de datos | Silver / Diccionario |
| Sin duplicados ni tipos ambiguos en la muestra | Igual se implementan; idempotencia se prueba por reproceso | Bronze / Tests |
| Volumen chico (~43k eventos, ~13 MB) | Justificar Big Data por **variedad, velocidad y veracidad**, y por escalabilidad del diseño, no por volumen actual | Análisis 5V |

### Preguntas abiertas para validar con el docente
1. ¿El desorden temporal de los archivos de eventos es intencional?
2. Subtotales negativos en billing: ¿notas de crédito válidas o error?
3. `exchange_rate_to_usd` ≠ 1 para USD: ¿qué valor prevalece?
4. Escala de `csat` (observado 0–7).